In [294]:
import warnings
warnings.filterwarnings('ignore')

In [295]:
import pandas as pd
import numpy as np
import scipy
import matplotlib.pyplot as plt

from rapidfuzz import fuzz, process
import ftfy

In [296]:
res = pd.read_csv("data/1976-2024-senate-state.csv", encoding='utf-8')
res.head()

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,special,candidate,party_detailed,writein,mode,candidatevotes,totalvotes,unofficial,version,party_simplified
0,1976,ARIZONA,AZ,4,86.0,61,US SENATE,statewide,gen,False,ALLAN NORWITZ,LIBERTARIAN,False,total,7310.0,741210.0,False,20210114,LIBERTARIAN
1,1976,ARIZONA,AZ,4,86.0,61,US SENATE,statewide,gen,False,BOB FIELD,INDEPENDENT,False,total,10765.0,741210.0,False,20210114,OTHER
2,1976,ARIZONA,AZ,4,86.0,61,US SENATE,statewide,gen,False,DENNIS DECONCINI,DEMOCRAT,False,total,400334.0,741210.0,False,20210114,DEMOCRAT
3,1976,ARIZONA,AZ,4,86.0,61,US SENATE,statewide,gen,False,SAM STEIGER,REPUBLICAN,False,total,321236.0,741210.0,False,20210114,REPUBLICAN
4,1976,ARIZONA,AZ,4,86.0,61,US SENATE,statewide,gen,False,WM. MATHEWS FEIGHAN,INDEPENDENT,False,total,1565.0,741210.0,False,20210114,OTHER


In [297]:
res.columns.values

array(['year', 'state', 'state_po', 'state_fips', 'state_cen', 'state_ic',
       'office', 'district', 'stage', 'special', 'candidate',
       'party_detailed', 'writein', 'mode', 'candidatevotes',
       'totalvotes', 'unofficial', 'version', 'party_simplified'],
      dtype=object)

In [298]:
res.shape

(3945, 19)

In [299]:
res['stage'].value_counts()

stage
gen           3616
GEN            314
pre              9
runoff           4
GEN RUNOFF       2
Name: count, dtype: int64

In [300]:
res[(res['state_po'] == 'GA') & (res['year'] == 2020)]

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,special,candidate,party_detailed,writein,mode,candidatevotes,totalvotes,unofficial,version,party_simplified
3461,2020,GEORGIA,GA,13,58.0,44,US SENATE,statewide,gen,False,DAVID A. PERDUE,REPUBLICAN,False,total,2462617.0,4952175.0,False,20210114,REPUBLICAN
3462,2020,GEORGIA,GA,13,58.0,44,US SENATE,statewide,gen,False,JON OSSOFF,DEMOCRAT,False,total,2374519.0,4952175.0,False,20210114,DEMOCRAT
3463,2020,GEORGIA,GA,13,58.0,44,US SENATE,statewide,gen,False,SHANE HAZEL,LIBERTARIAN,False,total,115039.0,4952175.0,False,20210114,LIBERTARIAN
3464,2020,GEORGIA,GA,13,58.0,44,US SENATE,statewide,gen,True,A. WAYNE JOHNSON,REPUBLICAN,False,total,36176.0,4914361.0,False,20210114,REPUBLICAN
3465,2020,GEORGIA,GA,13,58.0,44,US SENATE,statewide,gen,True,AL BARTELL,INDEPENDENT,False,total,14640.0,4914361.0,False,20210114,OTHER
3466,2020,GEORGIA,GA,13,58.0,44,US SENATE,statewide,gen,True,ALLEN BUCKLEY,INDEPENDENT,False,total,17954.0,4914361.0,False,20210114,OTHER
3467,2020,GEORGIA,GA,13,58.0,44,US SENATE,statewide,gen,True,ANNETTE DAVIS JACKSON,REPUBLICAN,False,total,44335.0,4914361.0,False,20210114,REPUBLICAN
3468,2020,GEORGIA,GA,13,58.0,44,US SENATE,statewide,gen,True,BRIAN SLOWINSKI,LIBERTARIAN,False,total,35431.0,4914361.0,False,20210114,LIBERTARIAN
3469,2020,GEORGIA,GA,13,58.0,44,US SENATE,statewide,gen,True,DEBORAH JACKSON,DEMOCRAT,False,total,324118.0,4914361.0,False,20210114,DEMOCRAT
3470,2020,GEORGIA,GA,13,58.0,44,US SENATE,statewide,gen,True,DERRICK E. GRAYSON,REPUBLICAN,False,total,51592.0,4914361.0,False,20210114,REPUBLICAN


In [301]:
res['mode'] = res['mode'].astype(str).map(lambda x: x.upper())
res['stage'] = res['stage'].astype(str).map(lambda x: x.upper())

In [302]:
ak_res_rcv = pd.read_csv('data/alaska_rcv_maximum_round_senate_results.csv')
ak_res_rcv.head()

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,runoff,special,candidate,party_detailed,writein,mode,candidatevotes,totalvotes,unofficial,version,party_simplified
0,2022,ALASKA,AK,2,94,81,US SENATE,statewide,GEN,NaN,False,LISA MURKOWSKI,REPUBLICAN,False,TOTAL,136330,253864,False,20230920,REPUBLICAN
1,2022,ALASKA,AK,2,94,81,US SENATE,statewide,GEN,NaN,False,KELLY C. TSHIBAKA,REPUBLICAN,False,TOTAL,117534,253864,False,20230920,REPUBLICAN


In [303]:
# 2022 and 2024 Alaska at-large district results are plurality (first round) vote, not maximum round RCV
# We want to train with maximum round
# Replace with max round results
# File is manually inputted; results from Wikipedia

ak_mask = ((res['state'] == 'ALASKA') &
          (res['year'] >= 2022))

res = res[~ak_mask]
res = pd.concat([res, ak_res_rcv], axis=0)

In [304]:
# Fix text encoding errors
res['candidate'] = res['candidate'].astype(str).map(ftfy.fix_text)

In [305]:
cutoff_year = 2014
res = res[(res['year'] >= cutoff_year) & (res['mode'] == 'TOTAL')]
res.head()

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,special,candidate,party_detailed,writein,mode,candidatevotes,totalvotes,unofficial,version,party_simplified,runoff
2861,2014,ALABAMA,AL,1,63.0,41,US SENATE,statewide,GEN,False,JEFF SESSIONS,REPUBLICAN,False,TOTAL,795606.0,818090.0,False,20210114,REPUBLICAN,NaN
2862,2014,ALABAMA,AL,1,63.0,41,US SENATE,statewide,GEN,False,nan,NaN,True,TOTAL,22484.0,818090.0,False,20210114,OTHER,NaN
2863,2014,ALASKA,AK,2,94.0,81,US SENATE,statewide,GEN,False,DAN SULLIVAN,REPUBLICAN,False,TOTAL,135445.0,282400.0,False,20210114,REPUBLICAN,NaN
2864,2014,ALASKA,AK,2,94.0,81,US SENATE,statewide,GEN,False,MARK BEGICH,DEMOCRAT,False,TOTAL,129431.0,282400.0,False,20210114,DEMOCRAT,NaN
2865,2014,ALASKA,AK,2,94.0,81,US SENATE,statewide,GEN,False,MARK S. FISH,LIBERTARIAN,False,TOTAL,10512.0,282400.0,False,20210114,LIBERTARIAN,NaN


In [306]:
# res = res[~(res['candidate'] == 'DAVID ALEXANDER')]

In [307]:
# # Handling fusion voting ## No need to for Senate: party_simplified is there
# resdem = res[res['party'] == 'DEMOCRAT']
# resrep = res[res['party'] == 'REPUBLICAN']
# res3rd = res[~res['party'].isin(['DEMOCRAT', 'REPUBLICAN'])]
# res = pd.concat([resdem, resrep, res3rd], axis=0)

# res_fusion = res.copy() #res[res['fusion_ticket']]

# res_fusion = res_fusion.groupby(['candidate', 'year', 'state', 'state_po', 'state_fips',
#                                 'state_cen', 'state_ic', 'office', 'district']).agg({
#     'stage':'first', 'runoff': 'first', 'special': 'first', 'party': 'first', 'writein': 'first', 'mode': 'first',
#     'totalvotes': 'first', 'unofficial': 'first', 'version': 'first',
#     'candidatevotes': 'sum'})

# # res_notfusion = res[~res['fusion_ticket']]
# res = res_fusion.reset_index() # pd.concat([res_fusion.reset_index(), res_notfusion])

In [308]:
## Handle Georgia runoffs
## We want to train on the head-to-head runoff elections, not the GEN before
res_ga = res[(res['state_po'] == 'GA') &
             (res['year'].isin([2021, 2022]))]
res = res[~((res['state_po'] == 'GA') &
             (res['year'].isin([2020, 2021, 2022])))]
res_ga = res_ga[res_ga['stage'].isin(['RUNOFF', 'GEN RUNOFF'])]
res = pd.concat([res, res_ga], axis=0)
res['year'] = res['year'].map(lambda x: 2020 if x == 2021 else x) ## For training purposes
res.shape

(1055, 20)

In [309]:
res[(res['state_po'] == 'GA') & (res['year'] == 2020)]

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,special,candidate,party_detailed,writein,mode,candidatevotes,totalvotes,unofficial,version,party_simplified,runoff
3625,2020,GEORGIA,GA,13,58.0,44,US SENATE,statewide,RUNOFF,False,DAVID A. PERDUE,REPUBLICAN,False,TOTAL,2213979.0,4483241.0,True,20210114,REPUBLICAN,NaN
3626,2020,GEORGIA,GA,13,58.0,44,US SENATE,statewide,RUNOFF,False,JON OSSOFF,DEMOCRAT,False,TOTAL,2269262.0,4483241.0,True,20210114,DEMOCRAT,NaN
3627,2020,GEORGIA,GA,13,58.0,44,US SENATE,statewide,RUNOFF,True,KELLY LOEFFLER,REPUBLICAN,False,TOTAL,2194848.0,4483294.0,True,20210114,REPUBLICAN,NaN
3628,2020,GEORGIA,GA,13,58.0,44,US SENATE,statewide,RUNOFF,True,RAPHAEL WARNOCK,DEMOCRAT,False,TOTAL,2288446.0,4483294.0,True,20210114,DEMOCRAT,NaN


In [310]:
res[(res['state_po'] == 'NE') & (res['year'] == 2024)]

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,special,candidate,party_detailed,writein,mode,candidatevotes,totalvotes,unofficial,version,party_simplified,runoff
3864,2024,NEBRASKA,NE,31,46.0,35,US SENATE,statewide,GEN,False,DAN OSBORN,BY PETITION,False,TOTAL,436493.0,938336.0,False,11/20/25,OTHER,NaN
3865,2024,NEBRASKA,NE,31,46.0,35,US SENATE,statewide,GEN,False,DEB FISCHER,REPUBLICAN,False,TOTAL,499124.0,938336.0,False,11/20/25,REPUBLICAN,NaN
3866,2024,NEBRASKA,NE,31,46.0,35,US SENATE,statewide,GEN,False,OTHER,OTHER,False,TOTAL,2719.0,938336.0,False,11/20/25,OTHER,NaN
3867,2024,NEBRASKA,NE,31,46.0,35,US SENATE,statewide,GEN,True,PETE RICKETTS,REPUBLICAN,False,TOTAL,585103.0,935005.0,True,20241212,REPUBLICAN,NaN
3868,2024,NEBRASKA,NE,31,46.0,35,US SENATE,statewide,GEN,True,PRESTON LOVE JR.,DEMOCRAT,False,TOTAL,349902.0,935005.0,True,20241212,DEMOCRAT,NaN


In [311]:
# Fix some discrepancies between party_detailed and party_simplified
res['party_simplified'] = res[['party_detailed', 'party_simplified']].apply(lambda x: 'DEMOCRAT' if x['party_detailed'] == 'DEMOCRATIC' else x['party_simplified'], axis=1)

In [312]:
# Handling significant independents
sign_inds = ['BERNIE SANDERS', 'ANGUS S. KING JR.', 'EVAN MCMULLIN', 'DAN OSBORN']
res['party_simplified'] = res[['candidate', 'party_simplified']].apply(lambda x: 'DEMOCRAT' if x['candidate'] in sign_inds else x['party_simplified'],
                                                                      axis=1)

In [313]:
# Focus on two party vote share in the model - no need to wrangle with third parties for now
res = res[res['party_simplified'].isin(['DEMOCRAT', 'REPUBLICAN'])]
res.head()

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,special,candidate,party_detailed,writein,mode,candidatevotes,totalvotes,unofficial,version,party_simplified,runoff
2861,2014,ALABAMA,AL,1,63.0,41,US SENATE,statewide,GEN,False,JEFF SESSIONS,REPUBLICAN,False,TOTAL,795606.0,818090.0,False,20210114,REPUBLICAN,NaN
2863,2014,ALASKA,AK,2,94.0,81,US SENATE,statewide,GEN,False,DAN SULLIVAN,REPUBLICAN,False,TOTAL,135445.0,282400.0,False,20210114,REPUBLICAN,NaN
2864,2014,ALASKA,AK,2,94.0,81,US SENATE,statewide,GEN,False,MARK BEGICH,DEMOCRAT,False,TOTAL,129431.0,282400.0,False,20210114,DEMOCRAT,NaN
2869,2014,ARKANSAS,AR,5,71.0,42,US SENATE,statewide,GEN,False,MARK L. PRYOR,DEMOCRAT,False,TOTAL,334174.0,847505.0,False,20210114,DEMOCRAT,NaN
2871,2014,ARKANSAS,AR,5,71.0,42,US SENATE,statewide,GEN,False,TOM COTTON,REPUBLICAN,False,TOTAL,478819.0,847505.0,False,20210114,REPUBLICAN,NaN


In [314]:
# To make life easier later wrt fuzzy string matching
# res = res.replace({
#     'LIZZIE FLETCHER': 'ELIZABETH FLETCHER',
# })

In [315]:
# res_12 = res[res['year'] == 2012]
# res = res[res['year'] >= cutoff_year-2]

In [316]:
# res_12.shape

In [317]:
# data_12 = pd.pivot_table(data=res_12, values='candidatevotes', columns=['party'], index=['year', 'state', 'state_po', 'special', 'district'], aggfunc='sum').reset_index()
# data_12 = data_12.fillna(0)
# data_12 = data_12.rename({'DEMOCRAT': 'dem', 'REPUBLICAN': 'rep'}, axis=1)
# data_12['dem_2p_pct'] = data_12['dem'] / (data_12['dem'] + data_12['rep']) * 100
# data_12.to_csv('transformed/house_res_2012.csv')
# data_12.head()

In [318]:
res.shape

(455, 20)

## FEC Data Wrangling

FEC data from: https://www.fec.gov/data/browse-data/?tab=bulk-data

In [319]:
fec_webl_colnames = ["CAND_ID", "CAND_NAME", "CAND_ICI", "PTY_CD", "CAND_PTY_AFFILIATION", "TTL_RECEIPTS", "TRANS_FROM_AUTH", "TTL_DISB", "TRANS_TO_AUTH", "COH_BOP", "COH_COP", "CAND_CONTRIB", "CAND_LOANS", "OTHER_LOANS", "CAND_LOAN_REPAY", "OTHER_LOAN_REPAY", "DEBTS_OWED_BY", "TTL_INDIV_CONTRIB", "CAND_OFFICE_ST", "CAND_OFFICE_DISTRICT", "SPEC_ELECTION", "PRIM_ELECTION", "RUN_ELECTION", "GEN_ELECTION", "GEN_ELECTION_PRECENT", "OTHER_POL_CMTE_CONTRIB", "POL_PTY_CONTRIB", "CVG_END_DT", "INDIV_REFUNDS", "CMTE_REFUNDS"]

In [320]:
webl12 = pd.read_table('data/fec/webl12.txt', sep='|', names=fec_webl_colnames)
webl14 = pd.read_table('data/fec/webl14.txt', sep='|', names=fec_webl_colnames)
webl16 = pd.read_table('data/fec/webl16.txt', sep='|', names=fec_webl_colnames)
webl18 = pd.read_table('data/fec/webl18.txt', sep='|', names=fec_webl_colnames)
webl20 = pd.read_table('data/fec/webl20.txt', sep='|', names=fec_webl_colnames)
webl22 = pd.read_table('data/fec/webl22.txt', sep='|', names=fec_webl_colnames)
webl24 = pd.read_table('data/fec/webl24.txt', sep='|', names=fec_webl_colnames)
webl24.head()

,CAND_ID,CAND_NAME,CAND_ICI,PTY_CD,CAND_PTY_AFFILIATION,TTL_RECEIPTS,TRANS_FROM_AUTH,TTL_DISB,TRANS_TO_AUTH,COH_BOP,COH_COP,CAND_CONTRIB,CAND_LOANS,OTHER_LOANS,CAND_LOAN_REPAY,OTHER_LOAN_REPAY,DEBTS_OWED_BY,TTL_INDIV_CONTRIB,CAND_OFFICE_ST,CAND_OFFICE_DISTRICT,SPEC_ELECTION,PRIM_ELECTION,RUN_ELECTION,GEN_ELECTION,GEN_ELECTION_PRECENT,OTHER_POL_CMTE_CONTRIB,POL_PTY_CONTRIB,CVG_END_DT,INDIV_REFUNDS,CMTE_REFUNDS
0,H2AK01158,"PELTOLA, MARY",I,1,DEM,13443537.46,951851.88,14050828.27,0.00,691260.30,83969.49,0.00,0.0,0.0,0.00,0.0,0.00,10800004.13,AK,0.0,NaN,NaN,NaN,NaN,NaN,1615986.30,9969.28,12/31/2024,161309.36,5625.0
1,H2AK01083,"BEGICH, NICHOLAS III",C,2,REP,2810467.65,176570.41,2747371.58,17659.66,41233.99,104330.06,599.00,0.0,0.0,25000.00,0.0,425000.00,2309088.23,AK,0.0,NaN,NaN,NaN,NaN,NaN,318750.00,5000.00,12/31/2024,23031.99,0.0
2,H4AK00156,"DAHLSTROM, NANCY",C,2,REP,996163.60,435712.04,790351.61,0.00,0.00,205811.99,0.00,0.0,0.0,0.00,0.0,0.00,286828.61,AK,0.0,NaN,NaN,NaN,NaN,NaN,262916.02,0.00,12/31/2024,3218.30,0.0
3,H4AL01255,"HOLMES, THOMAS BETHUNE MR.",C,1,DEM,17698.86,0.00,16817.50,0.00,0.00,410.16,10115.56,2500.0,0.0,0.00,0.0,0.00,3076.82,AL,1.0,NaN,NaN,NaN,NaN,NaN,2002.63,0.00,12/31/2024,0.00,0.0
4,H0AL01055,"CARL, JERRY LEE, JR",I,2,REP,2246839.19,547807.76,2631446.59,27316.59,453897.82,69290.42,0.00,0.0,0.0,176202.71,0.0,254535.87,1063039.38,AL,1.0,NaN,NaN,NaN,NaN,NaN,634500.00,0.00,12/31/2024,193499.00,84500.0


In [321]:
fec_cn_colnames = ["CAND_ID", "CAND_NAME", "CAND_PTY_AFFILIATION", "CAND_ELECTION_YR", "CAND_OFFICE_ST", "CAND_OFFICE", "CAND_OFFICE_DISTRICT", "CAND_ICI", "CAND_STATUS", "CAND_PCC", "CAND_ST1", "CAND_ST2", "CAND_CITY", "CAND_ST", "CAND_ZIP"]

cn12 = pd.read_table('data/fec/cn12.txt', sep='|', names=fec_cn_colnames)
cn14 = pd.read_table('data/fec/cn14.txt', sep='|', names=fec_cn_colnames)
cn16 = pd.read_table('data/fec/cn16.txt', sep='|', names=fec_cn_colnames)
cn18 = pd.read_table('data/fec/cn18.txt', sep='|', names=fec_cn_colnames)
cn20 = pd.read_table('data/fec/cn20.txt', sep='|', names=fec_cn_colnames)
cn22 = pd.read_table('data/fec/cn22.txt', sep='|', names=fec_cn_colnames)
cn24 = pd.read_table('data/fec/cn24.txt', sep='|', names=fec_cn_colnames)
cn24.head()

,CAND_ID,CAND_NAME,CAND_PTY_AFFILIATION,CAND_ELECTION_YR,CAND_OFFICE_ST,CAND_OFFICE,CAND_OFFICE_DISTRICT,CAND_ICI,CAND_STATUS,CAND_PCC,CAND_ST1,CAND_ST2,CAND_CITY,CAND_ST,CAND_ZIP
0,H0AK00105,"LAMB, THOMAS",NNE,2020,AK,H,0.0,C,N,C00607515,1861 W LAKE LUCILLE DR,NaN,WASILLA,AK,99654
1,H0AL01055,"CARL, JERRY LEE, JR",REP,2024,AL,H,1.0,I,C,C00697789,PO BOX 852138,NaN,MOBILE,AL,36685
2,H0AL01097,"AVERHART, JAMES",DEM,2024,AL,H,2.0,C,C,C00708867,811 SPRINGHILL AV,NaN,MOBILE,AL,36602
3,H0AL02087,"ROBY, MARTHA",REP,2020,AL,H,2.0,I,P,C00462143,NaN,NaN,MONTGOMERY,NaN,NaN
4,H0AL02137,"DISMUKES, WILL",REP,2020,AL,H,2.0,O,P,C00714337,PO BOX 6811188,NaN,PRATTVILLE,AL,36068


In [322]:
def wrangle_fec(webl, cn, year):
    webl['cand_name_lst'] = webl['CAND_NAME'].str.split(',')
    cn['cand_name_lst'] = cn['CAND_NAME'].str.split(',')

    def refactor_str(cand_lst):
        if len(cand_lst) == 3:
            return cand_lst[1] + ' ' + cand_lst[0] + ' ' + cand_lst[2]
        elif len(cand_lst) == 1:
            return cand_lst[0]
        else:
            return cand_lst[1] + ' ' + cand_lst[0]

    webl['cand'] = webl['cand_name_lst'].map(refactor_str)
    cn['cand'] = cn['cand_name_lst'].map(refactor_str)

    df = pd.merge(left=webl, right=cn, on='CAND_ID', how='inner')
    df = df[[col for col in df.columns.values if ('_y' not in col)]]

    df.columns = df.columns.str.strip('_x')

    df['year'] = np.full(shape=(df.shape[0],), fill_value=year)

    return df

In [323]:
fec12 = wrangle_fec(webl12, cn12, 2012)
fec14 = wrangle_fec(webl14, cn14, 2014)
fec16 = wrangle_fec(webl16, cn16, 2016)
fec18 = wrangle_fec(webl18, cn18, 2018)
fec20 = wrangle_fec(webl20, cn20, 2020)
fec22 = wrangle_fec(webl22, cn22, 2022)
fec24 = wrangle_fec(webl24, cn24, 2024)
fec24.head()

,CAND_ID,CAND_NAME,CAND_ICI,PTY_CD,CAND_PTY_AFFILIATION,TTL_RECEIPTS,TRANS_FROM_AUTH,TTL_DISB,TRANS_TO_AUTH,COH_BOP,COH_COP,CAND_CONTRIB,CAND_LOANS,OTHER_LOANS,CAND_LOAN_REPAY,OTHER_LOAN_REPAY,DEBTS_OWED_BY,TTL_INDIV_CONTRIB,CAND_OFFICE_ST,CAND_OFFICE_DISTRICT,SPEC_ELECTION,PRIM_ELECTION,RUN_ELECTION,GEN_ELECTION,GEN_ELECTION_PRECENT,OTHER_POL_CMTE_CONTRIB,POL_PTY_CONTRIB,CVG_END_DT,INDIV_REFUNDS,CMTE_REFUNDS,cand_name_lst,cand,CAND_ELECTION_YR,CAND_OFFICE,CAND_STATUS,CAND_PCC,CAND_ST1,CAND_ST2,CAND_CITY,CAND_ST,CAND_ZIP,year
0,H2AK01158,"PELTOLA, MARY",I,1,DEM,13443537.46,951851.88,14050828.27,0.00,691260.30,83969.49,0.00,0.0,0.0,0.00,0.0,0.00,10800004.13,AK,0.0,NaN,NaN,NaN,NaN,NaN,1615986.30,9969.28,12/31/2024,161309.36,5625.0,"[PELTOLA, MARY]",MARY PELTOLA,2024,H,C,C00812388,810 N STREET,SUITE 301,ANCHORAGE,AK,99501,2024
1,H2AK01083,"BEGICH, NICHOLAS III",C,2,REP,2810467.65,176570.41,2747371.58,17659.66,41233.99,104330.06,599.00,0.0,0.0,25000.00,0.0,425000.00,2309088.23,AK,0.0,NaN,NaN,NaN,NaN,NaN,318750.00,5000.00,12/31/2024,23031.99,0.0,"[BEGICH, NICHOLAS III]",NICHOLAS III BEGICH,2024,H,C,C00792341,PO BOX 671710,NaN,CHUGIAK,AK,99567,2024
2,H4AK00156,"DAHLSTROM, NANCY",C,2,REP,996163.60,435712.04,790351.61,0.00,0.00,205811.99,0.00,0.0,0.0,0.00,0.0,0.00,286828.61,AK,0.0,NaN,NaN,NaN,NaN,NaN,262916.02,0.00,12/31/2024,3218.30,0.0,"[DAHLSTROM, NANCY]",NANCY DAHLSTROM,2024,H,C,C00856716,PO BOX 242442,NaN,ANCHORAGE,AK,99524,2024
3,H4AL01255,"HOLMES, THOMAS BETHUNE MR.",C,1,DEM,17698.86,0.00,16817.50,0.00,0.00,410.16,10115.56,2500.0,0.0,0.00,0.0,0.00,3076.82,AL,1.0,NaN,NaN,NaN,NaN,NaN,2002.63,0.00,12/31/2024,0.00,0.0,"[HOLMES, THOMAS BETHUNE MR.]",THOMAS BETHUNE MR. HOLMES,2024,H,C,C00866939,"2117 CHARINGWOOD DRIVE WEST, MOBIL",NaN,MOBILE,AL,366952916,2024
4,H0AL01055,"CARL, JERRY LEE, JR",I,2,REP,2246839.19,547807.76,2631446.59,27316.59,453897.82,69290.42,0.00,0.0,0.0,176202.71,0.0,254535.87,1063039.38,AL,1.0,NaN,NaN,NaN,NaN,NaN,634500.00,0.00,12/31/2024,193499.00,84500.0,"[CARL, JERRY LEE, JR]",JERRY LEE CARL JR,2024,H,C,C00697789,PO BOX 852138,NaN,MOBILE,AL,36685,2024


In [324]:
fec = pd.concat([fec12, fec14, fec16, fec18, fec20, fec22, fec24], axis=0)
fec.head()

,CAND_ID,CAND_NAME,CAND_ICI,PTY_CD,CAND_PTY_AFFILIATION,TTL_RECEIPTS,TRANS_FROM_AUTH,TTL_DISB,TRANS_TO_AUTH,COH_BOP,COH_COP,CAND_CONTRIB,CAND_LOANS,OTHER_LOANS,CAND_LOAN_REPAY,OTHER_LOAN_REPAY,DEBTS_OWED_BY,TTL_INDIV_CONTRIB,CAND_OFFICE_ST,CAND_OFFICE_DISTRICT,SPEC_ELECTION,PRIM_ELECTION,RUN_ELECTION,GEN_ELECTION,GEN_ELECTION_PRECENT,OTHER_POL_CMTE_CONTRIB,POL_PTY_CONTRIB,CVG_END_DT,INDIV_REFUNDS,CMTE_REFUNDS,cand_name_lst,cand,CAND_ELECTION_YR,CAND_OFFICE,CAND_STATUS,CAND_PCC,CAND_ST1,CAND_ST2,CAND_CITY,CAND_ST,CAND_ZIP,year
0,H2AK00093,"URQUIDI, DOUGLAS C",C,1,DEM,3208.82,0.0,3089.84,0.00,0.0,118.98,2635.48,0.00,0.0,0.00,0.0,2520.82,573.34,AK,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,09/30/2012,0.0,0.0,"[URQUIDI, DOUGLAS C]",DOUGLAS C URQUIDI,2012,H,N,C00515767,12134 COPPER MT DR,NaN,EAGLE RIVER,AK,99577.0,2012
1,H2AK00101,"MOORE, MATTHEW EDWARD",C,1,DEM,43184.98,0.0,43184.98,0.00,0.0,0.00,4538.56,33515.99,0.0,278.02,0.0,0.00,5130.00,AK,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,12/31/2012,0.0,0.0,"[MOORE, MATTHEW EDWARD]",MATTHEW EDWARD MOORE,2012,H,C,C00520544,7035 TULUGAK CIRCLE,NaN,ANCHORAGE,AK,99507.0,2012
2,H2AK00119,"CISSNA, SHARON MARIE",C,1,DEM,19660.00,0.0,24388.00,1450.00,0.0,0.00,0.00,17842.00,0.0,0.00,0.0,13000.00,13818.00,AK,0.0,NaN,W,NaN,L,28.0,1000.0,0.0,12/31/2012,0.0,0.0,"[CISSNA, SHARON MARIE]",SHARON MARIE CISSNA,2012,H,C,NaN,2612 EAST 20TH,NaN,ANCHORAGE,AK,99508.0,2012
3,H2AK00127,"CHESNUT, DEBRA SUE",C,1,DEM,16694.00,0.0,16247.00,91.16,0.0,91.00,7700.00,0.00,0.0,0.00,0.0,0.00,8994.00,AK,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,12/31/2012,2000.0,0.0,"[CHESNUT, DEBRA SUE]",DEBRA SUE CHESNUT,2012,H,C,C00523662,PO BOX 81456,NaN,FAIRBANKS,AK,99708.0,2012
4,H4AK00057,"VONDERSAAR, FRANK J",C,1,DEM,1109.00,0.0,1109.59,0.00,0.0,0.00,0.00,1100.00,0.0,84.00,0.0,0.00,9.00,AK,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,08/29/2012,0.0,0.0,"[VONDERSAAR, FRANK J]",FRANK J VONDERSAAR,2012,H,N,C00503896,1740 SALTWATER DR,NaN,HOMER,AK,996038321.0,2012


In [325]:
fec24.shape, fec.shape

((2373, 42), (16444, 42))

In [326]:
fec.columns.values

array(['CAND_ID', 'CAND_NAME', 'CAND_ICI', 'PTY_CD',
       'CAND_PTY_AFFILIATION', 'TTL_RECEIPTS', 'TRANS_FROM_AUTH',
       'TTL_DISB', 'TRANS_TO_AUTH', 'COH_BOP', 'COH_COP', 'CAND_CONTRIB',
       'CAND_LOANS', 'OTHER_LOANS', 'CAND_LOAN_REPAY', 'OTHER_LOAN_REPAY',
       'DEBTS_OWED_BY', 'TTL_INDIV_CONTRIB', 'CAND_OFFICE_ST',
       'CAND_OFFICE_DISTRICT', 'SPEC_ELECTION', 'PRIM_ELECTION',
       'RUN_ELECTION', 'GEN_ELECTION', 'GEN_ELECTION_PRECENT',
       'OTHER_POL_CMTE_CONTRIB', 'POL_PTY_CONTRIB', 'CVG_END_DT',
       'INDIV_REFUNDS', 'CMTE_REFUNDS', 'cand_name_lst', 'cand',
       'CAND_ELECTION_YR', 'CAND_OFFICE', 'CAND_STATUS', 'CAND_PCC',
       'CAND_ST1', 'CAND_ST2', 'CAND_CITY', 'CAND_ST', 'CAND_ZIP', 'year'],
      dtype=object)

In [327]:
fuzz.partial_ratio('JAMAAL BOWMAN', 'GEORGE LATIMER')

21.052631578947366

In [328]:
fuzz.token_sort_ratio('ERIC MICHAEL SWALWELL', 'ERIC SWALWELL')

76.47058823529412

In [329]:
fec = fec[fec['CAND_OFFICE'] == 'S']

In [330]:
def get_fuzzymatch_cand(year, state_po, party, candidate):
    '''
    :param party: Party name as noted in `res` dataframe.
    :param candidate: Candidate name as noted in the `res` dataframe.
    '''

    if party == 'DEMOCRAT':
        fec_party = 'DEM'
    elif party == 'REPUBLICAN':
        fec_party = 'REP'
    
    df = fec[(fec['year'] == year) &
        (fec['CAND_OFFICE_ST'] == state_po)]

    resdf = res[(res['year'] == year) &
        (res['state_po'] == state_po) &
        (res['candidate'] == candidate)]

    if party == 'DEM':
        cand_col_str = 'dem_cand'
    else:
        cand_col_str = 'rep_cand'

    # df['cand_fuzzymatch'] = df['CAND_NAME'].str.title().apply(
    #         lambda x: process.extractOne(x, mit_df[cand_col_str].values[0], scorer=fuzz.partial_ratio)[0]
    # )
    if resdf.shape[0] == 0:
        return ''

    if df.shape[0] == 0:
        #  raise ValueError('Candidate available in MIT Election Lab dataset, but not available in FEC dataset.')
        return 'Error'
    
    fuzzymatch = process.extractOne(resdf['candidate'].values[0], df['CAND_NAME'].values, scorer=fuzz.WRatio, score_cutoff=55)
    return fuzzymatch if fuzzymatch is None else fuzzymatch[0]

In [331]:
res['fec_candidate_name'] = res.apply(
    lambda x: get_fuzzymatch_cand(x['year'], x['state_po'], x['party_simplified'], x['candidate']),
    axis=1
)
res.head()

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,special,candidate,party_detailed,writein,mode,candidatevotes,totalvotes,unofficial,version,party_simplified,runoff,fec_candidate_name
2861,2014,ALABAMA,AL,1,63.0,41,US SENATE,statewide,GEN,False,JEFF SESSIONS,REPUBLICAN,False,TOTAL,795606.0,818090.0,False,20210114,REPUBLICAN,NaN,"SESSIONS, JEFF"
2863,2014,ALASKA,AK,2,94.0,81,US SENATE,statewide,GEN,False,DAN SULLIVAN,REPUBLICAN,False,TOTAL,135445.0,282400.0,False,20210114,REPUBLICAN,NaN,"SULLIVAN, DAN"
2864,2014,ALASKA,AK,2,94.0,81,US SENATE,statewide,GEN,False,MARK BEGICH,DEMOCRAT,False,TOTAL,129431.0,282400.0,False,20210114,DEMOCRAT,NaN,"BEGICH, MARK"
2869,2014,ARKANSAS,AR,5,71.0,42,US SENATE,statewide,GEN,False,MARK L. PRYOR,DEMOCRAT,False,TOTAL,334174.0,847505.0,False,20210114,DEMOCRAT,NaN,"PRYOR, MARK LUNSFORD"
2871,2014,ARKANSAS,AR,5,71.0,42,US SENATE,statewide,GEN,False,TOM COTTON,REPUBLICAN,False,TOTAL,478819.0,847505.0,False,20210114,REPUBLICAN,NaN,"COTTON, THOMAS"


In [332]:
res.shape

(455, 21)

In [333]:
res[res['fec_candidate_name'] == 'Error']

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,special,candidate,party_detailed,writein,mode,candidatevotes,totalvotes,unofficial,version,party_simplified,runoff,fec_candidate_name


In [334]:
# Fixing some errors in the fuzzy string matching
# Below dictionary is Claude-generated because 455 rows is still a lot
corrections = {
    ('MARK P. MEUSER', 2022, 'CA'):        None,
    ('JOHN FLEMING', 2016, 'LA'):          'FLEMING, JOHN C JR',
    ('JIM RISCH', 2020, 'ID'):             'RISCH, JAMES E',
    ('JOHN CORNYN', 2020, 'TX'):           'CORNYN, JOHN SEN',
    ('MARSHA BLACKBURN', 2024, 'TN'):      'BLACKBURN, MARSHA MRS.',
    ('WILL BOYD', 2022, 'AL'):             'BOYD, WILLIE EUGENE DR. JR.',
    ('EDWARD J. MARKEY', 2020, 'MA'):      'MARKEY, EDWARD J. SEN.',
    ('CHARLES W. BOUSTANY, JR.', 2016, 'LA'): 'BOUSTANY, CHARLES W JR DR',
    ('WILLIAM P. WAYMIRE JR.', 2014, 'LA'): None,
    ('BILL CASSIDY', 2014, 'LA'):          'CASSIDY, WILLIAM',
    ('"BILL" CASSIDY', 2020, 'LA'):        'CASSIDY, WILLIAM M.',
    ('BOB HUGIN', 2018, 'NJ'):             'HUGIN, ROBERT',
    ('\\LUKE\\" MIXON"', 2022, 'LA'):      'MIXON, LUKE',
    ('MCDERMOTT, BOB', 2022, 'HI'):        None,
    ('LUCAS KUNCE', 2024, 'MO'):           None,
}


for (cand, year, state) in corrections.keys():
    mask = (
        (res['candidate'] == cand) &
        (res['year'] == year) &
        (res['state_po'] == state)
    )

    res.loc[mask, 'fec_candidate_name'] = corrections[(cand, year, state)]

res.head()

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,special,candidate,party_detailed,writein,mode,candidatevotes,totalvotes,unofficial,version,party_simplified,runoff,fec_candidate_name
2861,2014,ALABAMA,AL,1,63.0,41,US SENATE,statewide,GEN,False,JEFF SESSIONS,REPUBLICAN,False,TOTAL,795606.0,818090.0,False,20210114,REPUBLICAN,NaN,"SESSIONS, JEFF"
2863,2014,ALASKA,AK,2,94.0,81,US SENATE,statewide,GEN,False,DAN SULLIVAN,REPUBLICAN,False,TOTAL,135445.0,282400.0,False,20210114,REPUBLICAN,NaN,"SULLIVAN, DAN"
2864,2014,ALASKA,AK,2,94.0,81,US SENATE,statewide,GEN,False,MARK BEGICH,DEMOCRAT,False,TOTAL,129431.0,282400.0,False,20210114,DEMOCRAT,NaN,"BEGICH, MARK"
2869,2014,ARKANSAS,AR,5,71.0,42,US SENATE,statewide,GEN,False,MARK L. PRYOR,DEMOCRAT,False,TOTAL,334174.0,847505.0,False,20210114,DEMOCRAT,NaN,"PRYOR, MARK LUNSFORD"
2871,2014,ARKANSAS,AR,5,71.0,42,US SENATE,statewide,GEN,False,TOM COTTON,REPUBLICAN,False,TOTAL,478819.0,847505.0,False,20210114,REPUBLICAN,NaN,"COTTON, THOMAS"


In [335]:
res.shape, fec.shape

((455, 21), (1960, 42))

In [336]:
res_og = res.copy()

In [337]:
res = pd.merge(left=res, right=fec, left_on=['year', 'state_po', 'fec_candidate_name'], right_on=['year', 'CAND_OFFICE_ST', 'CAND_NAME'], how='left', 
               indicator=False)
res.head()

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,special,candidate,party_detailed,writein,mode,candidatevotes,totalvotes,unofficial,version,party_simplified,runoff,fec_candidate_name,CAND_ID,CAND_NAME,CAND_ICI,PTY_CD,CAND_PTY_AFFILIATION,TTL_RECEIPTS,TRANS_FROM_AUTH,TTL_DISB,TRANS_TO_AUTH,COH_BOP,COH_COP,CAND_CONTRIB,CAND_LOANS,OTHER_LOANS,CAND_LOAN_REPAY,OTHER_LOAN_REPAY,DEBTS_OWED_BY,TTL_INDIV_CONTRIB,CAND_OFFICE_ST,CAND_OFFICE_DISTRICT,SPEC_ELECTION,PRIM_ELECTION,RUN_ELECTION,GEN_ELECTION,GEN_ELECTION_PRECENT,OTHER_POL_CMTE_CONTRIB,POL_PTY_CONTRIB,CVG_END_DT,INDIV_REFUNDS,CMTE_REFUNDS,cand_name_lst,cand,CAND_ELECTION_YR,CAND_OFFICE,CAND_STATUS,CAND_PCC,CAND_ST1,CAND_ST2,CAND_CITY,CAND_ST,CAND_ZIP
0,2014,ALABAMA,AL,1,63.0,41,US SENATE,statewide,GEN,False,JEFF SESSIONS,REPUBLICAN,False,TOTAL,795606.0,818090.0,False,20210114,REPUBLICAN,NaN,"SESSIONS, JEFF",S6AL00195,"SESSIONS, JEFF",I,2.0,REP,1382806.00,129468.0,1335847.00,625000.0,2777227.00,2824187.0,0.0,0.0,0.0,0.0,0.0,0.0,381899.00,AL,0.0,NaN,W,NaN,NaN,NaN,825945.00,0.0,12/31/2014,0.00,0.0,"[SESSIONS, JEFF]",JEFF SESSIONS,2014.0,S,C,C00306704,PO BOX 4278,NaN,MONTGOMERY,AL,36103.0
1,2014,ALASKA,AK,2,94.0,81,US SENATE,statewide,GEN,False,DAN SULLIVAN,REPUBLICAN,False,TOTAL,135445.0,282400.0,False,20210114,REPUBLICAN,NaN,"SULLIVAN, DAN",S4AK00214,"SULLIVAN, DAN",C,2.0,REP,7977268.00,1122098.0,7797251.00,0.0,0.00,180029.0,600.0,0.0,0.0,0.0,0.0,0.0,5966351.00,AK,0.0,NaN,NaN,NaN,NaN,NaN,834683.00,45400.0,12/31/2014,57100.00,5000.0,"[SULLIVAN, DAN]",DAN SULLIVAN,2014.0,S,C,C00551093,3705 ARCTIC BOULEVARD #447,NaN,ANCHORAGE,AK,99503.0
2,2014,ALASKA,AK,2,94.0,81,US SENATE,statewide,GEN,False,MARK BEGICH,DEMOCRAT,False,TOTAL,129431.0,282400.0,False,20210114,DEMOCRAT,NaN,"BEGICH, MARK",S8AK00090,"BEGICH, MARK",I,1.0,DEM,8950661.00,362884.0,9883505.00,250.0,952260.00,19416.0,0.0,0.0,0.0,0.0,0.0,13500.0,6020171.00,AK,0.0,NaN,NaN,NaN,NaN,NaN,2546930.00,1000.0,12/31/2014,77866.00,7400.0,"[BEGICH, MARK]",MARK BEGICH,2014.0,S,C,C00458059,1231 W NORTHERN LIGHTS BLVD #605,NaN,ANCHORAGE,AK,99503.0
3,2014,ARKANSAS,AR,5,71.0,42,US SENATE,statewide,GEN,False,MARK L. PRYOR,DEMOCRAT,False,TOTAL,334174.0,847505.0,False,20210114,DEMOCRAT,NaN,"PRYOR, MARK LUNSFORD",S0AR00028,"PRYOR, MARK LUNSFORD",I,1.0,DEM,11616728.00,249334.0,13215385.00,250.0,1734136.00,135479.0,0.0,0.0,0.0,0.0,0.0,0.0,8000604.00,AR,0.0,NaN,W,NaN,NaN,NaN,3293383.00,45400.0,12/31/2014,69501.00,8504.0,"[PRYOR, MARK LUNSFORD]",MARK LUNSFORD PRYOR,2014.0,S,C,C00366401,PO BOX 2720,NaN,LITTLE ROCK,AR,72203.0
4,2014,ARKANSAS,AR,5,71.0,42,US SENATE,statewide,GEN,False,TOM COTTON,REPUBLICAN,False,TOTAL,478819.0,847505.0,False,20210114,REPUBLICAN,NaN,"COTTON, THOMAS",S4AR00103,"COTTON, THOMAS",C,2.0,REP,13911690.06,925209.0,13814166.59,0.0,118350.97,215874.0,0.0,0.0,0.0,0.0,0.0,447694.0,10892504.11,AR,0.0,NaN,W,NaN,NaN,NaN,1853167.95,65525.0,12/31/2014,111146.34,1000.0,"[COTTON, THOMAS]",THOMAS COTTON,2014.0,S,C,C00499988,PO BOX 379,NaN,DARDANELLE,AR,72834.0


In [338]:
res.shape

(456, 62)

In [339]:
pd.set_option('display.max_columns', 100)

In [341]:
res[res.duplicated(subset=['candidate', 'year', 'state_po', 'party_simplified'], keep=False)]

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,special,candidate,party_detailed,writein,mode,candidatevotes,totalvotes,unofficial,version,party_simplified,runoff,fec_candidate_name,CAND_ID,CAND_NAME,CAND_ICI,PTY_CD,CAND_PTY_AFFILIATION,TTL_RECEIPTS,TRANS_FROM_AUTH,TTL_DISB,TRANS_TO_AUTH,COH_BOP,COH_COP,CAND_CONTRIB,CAND_LOANS,OTHER_LOANS,CAND_LOAN_REPAY,OTHER_LOAN_REPAY,DEBTS_OWED_BY,TTL_INDIV_CONTRIB,CAND_OFFICE_ST,CAND_OFFICE_DISTRICT,SPEC_ELECTION,PRIM_ELECTION,RUN_ELECTION,GEN_ELECTION,GEN_ELECTION_PRECENT,OTHER_POL_CMTE_CONTRIB,POL_PTY_CONTRIB,CVG_END_DT,INDIV_REFUNDS,CMTE_REFUNDS,cand_name_lst,cand,CAND_ELECTION_YR,CAND_OFFICE,CAND_STATUS,CAND_PCC,CAND_ST1,CAND_ST2,CAND_CITY,CAND_ST,CAND_ZIP
254,2020,KANSAS,KS,20,47.0,32,US SENATE,statewide,GEN,False,ROGER MARSHALL,REPUBLICAN,False,TOTAL,727962.0,1367755.0,False,20210114,REPUBLICAN,NaN,"MARSHALL, ROGER W",H6KS01179,"MARSHALL, ROGER W",I,2.0,REP,6761143.55,184021.62,7159470.55,215.72,608285.86,209958.86,0.0,0.0,0.0,35000.0,0.0,0.0,4479222.33,KS,0.0,NaN,NaN,NaN,NaN,NaN,2034570.00,46450.00,12/31/2020,112955.13,13602.4,"[MARSHALL, ROGER W]",ROGER W MARSHALL,2020.0,S,C,C00576173,4501 QUAIL CREEK DR,NaN,GREAT BEND,KS,67530.0
255,2020,KANSAS,KS,20,47.0,32,US SENATE,statewide,GEN,False,ROGER MARSHALL,REPUBLICAN,False,TOTAL,727962.0,1367755.0,False,20210114,REPUBLICAN,NaN,"MARSHALL, ROGER W",S0KS00315,"MARSHALL, ROGER W",O,2.0,REP,6772872.34,187730.64,7171199.34,10718.64,608285.86,209958.86,0.0,0.0,0.0,35000.0,0.0,0.0,4487242.10,KS,0.0,NaN,NaN,NaN,NaN,NaN,2034570.00,46450.00,12/31/2020,112955.13,13602.4,"[MARSHALL, ROGER W]",ROGER W MARSHALL,2020.0,S,C,C00576173,4501 QUAIL CREEK DR,NaN,GREAT BEND,KS,67530.0
312,2022,CALIFORNIA,CA,6,93.0,71,US SENATE,statewide,GEN,False,ALEX PADILLA,DEMOCRAT,False,TOTAL,6621621.0,10843650.0,False,20230920,DEMOCRAT,NaN,"PADILLA, ALEX",S2CA00955,"PADILLA, ALEX",I,1.0,DEM,11784617.88,203172.87,4566222.01,0.00,234830.79,7453226.66,0.0,0.0,0.0,0.0,0.0,0.0,9352290.72,CA,0.0,NaN,NaN,NaN,NaN,NaN,2227671.43,376.85,12/31/2022,110350.07,11400.0,"[PADILLA, ALEX]",ALEX PADILLA,2022.0,S,C,C00765164,777 S FIGUEROA ST,SUITE 4050,LOS ANGELES,CA,90017.0
313,2022,CALIFORNIA,CA,6,93.0,71,US SENATE,statewide,GEN,False,MARK P. MEUSER,REPUBLICAN,False,TOTAL,4222029.0,10843650.0,False,20230920,REPUBLICAN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
314,2022,CALIFORNIA,CA,6,93.0,71,US SENATE,statewide,GEN,True,ALEX PADILLA,DEMOCRAT,False,TOTAL,6559308.0,10771758.0,False,20230920,DEMOCRAT,NaN,"PADILLA, ALEX",S2CA00955,"PADILLA, ALEX",I,1.0,DEM,11784617.88,203172.87,4566222.01,0.00,234830.79,7453226.66,0.0,0.0,0.0,0.0,0.0,0.0,9352290.72,CA,0.0,NaN,NaN,NaN,NaN,NaN,2227671.43,376.85,12/31/2022,110350.07,11400.0,"[PADILLA, ALEX]",ALEX PADILLA,2022.0,S,C,C00765164,777 S FIGUEROA ST,SUITE 4050,LOS ANGELES,CA,90017.0
315,2022,CALIFORNIA,CA,6,93.0,71,US SENATE,statewide,GEN,True,MARK P. MEUSER,REPUBLICAN,False,TOTAL,4212450.0,10771758.0,False,20230920,REPUBLICAN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
379,2024,CALIFORNIA,CA,6,93.0,71,US SENATE,statewide,GEN,False,ADAM B. SCHIFF,DEMOCRAT,False,TOTAL,9036252.0,15348846.0,False,11/20/25,DEMOCRAT,NaN,"SCHIFF, ADAM",S4CA00555,"SCHIFF, ADAM",O,1.0,DEM,48149196.78,536367.79,62791130.49,0.00,21022960.33,6381026.62,0.0,0.0,0.0,0.0,0.0,0.0,45110628.66,CA,0.0,NaN,NaN,NaN,NaN,NaN,1015453.06,250.00,12/31/2024,706974.81,22599.0,"[SCHIFF, ADAM]",ADAM SCHIFF,2024.0,S,C,C00343871,611 PENNSYLVANIA AVE SE,#143,WASHINGTON,DC,20003
380,2024,CALIFORNIA,CA,6,93.0,71,US SENATE,statewide,GEN,False,STEVE GARVEY,REPUBLICAN,False,TOTAL,6312594.0,15348846.0,False,11/20/25,REPUBLICAN,NaN,"GARVEY, STEVE",S4CA00803,"GARVEY, STEVE",O,2.0,REP,20356132.85,47761.39,19775163.62,0.00,0.

In [345]:
# Drop duplicates formed in merging
# Use 'first' here bc I'm pretty sure the "first" options for Calif and that one KS race are the correct one
# Don't just filter out unofficial results - for some races that's the only entry
res = res.drop_duplicates(subset=['candidate', 'year', 'state_po', 'party_simplified'], keep='first', ignore_index=True)
res.shape

(451, 62)

In [346]:
res.columns.values

array(['year', 'state', 'state_po', 'state_fips', 'state_cen', 'state_ic',
       'office', 'district', 'stage', 'special', 'candidate',
       'party_detailed', 'writein', 'mode', 'candidatevotes',
       'totalvotes', 'unofficial', 'version', 'party_simplified',
       'runoff', 'fec_candidate_name', 'CAND_ID', 'CAND_NAME', 'CAND_ICI',
       'PTY_CD', 'CAND_PTY_AFFILIATION', 'TTL_RECEIPTS',
       'TRANS_FROM_AUTH', 'TTL_DISB', 'TRANS_TO_AUTH', 'COH_BOP',
       'COH_COP', 'CAND_CONTRIB', 'CAND_LOANS', 'OTHER_LOANS',
       'CAND_LOAN_REPAY', 'OTHER_LOAN_REPAY', 'DEBTS_OWED_BY',
       'TTL_INDIV_CONTRIB', 'CAND_OFFICE_ST', 'CAND_OFFICE_DISTRICT',
       'SPEC_ELECTION', 'PRIM_ELECTION', 'RUN_ELECTION', 'GEN_ELECTION',
       'GEN_ELECTION_PRECENT', 'OTHER_POL_CMTE_CONTRIB',
       'POL_PTY_CONTRIB', 'CVG_END_DT', 'INDIV_REFUNDS', 'CMTE_REFUNDS',
       'cand_name_lst', 'cand', 'CAND_ELECTION_YR', 'CAND_OFFICE',
       'CAND_STATUS', 'CAND_PCC', 'CAND_ST1', 'CAND_ST2', 'CAND_CITY',


In [347]:
res[res.duplicated(subset=['year', 'state', 'special', 'candidate', 'TTL_INDIV_CONTRIB'])]

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,special,candidate,party_detailed,writein,mode,candidatevotes,totalvotes,unofficial,version,party_simplified,runoff,fec_candidate_name,CAND_ID,CAND_NAME,CAND_ICI,PTY_CD,CAND_PTY_AFFILIATION,TTL_RECEIPTS,TRANS_FROM_AUTH,TTL_DISB,TRANS_TO_AUTH,COH_BOP,COH_COP,CAND_CONTRIB,CAND_LOANS,OTHER_LOANS,CAND_LOAN_REPAY,OTHER_LOAN_REPAY,DEBTS_OWED_BY,TTL_INDIV_CONTRIB,CAND_OFFICE_ST,CAND_OFFICE_DISTRICT,SPEC_ELECTION,PRIM_ELECTION,RUN_ELECTION,GEN_ELECTION,GEN_ELECTION_PRECENT,OTHER_POL_CMTE_CONTRIB,POL_PTY_CONTRIB,CVG_END_DT,INDIV_REFUNDS,CMTE_REFUNDS,cand_name_lst,cand,CAND_ELECTION_YR,CAND_OFFICE,CAND_STATUS,CAND_PCC,CAND_ST1,CAND_ST2,CAND_CITY,CAND_ST,CAND_ZIP


In [348]:
res = res.drop_duplicates(subset=['state', 'year', 'candidate'], keep='last')
res.shape

(451, 62)

In [349]:
res['inc'] = res['CAND_ICI'].map(lambda x: True if x == 'I' else False)
res['inc'].value_counts()

inc
False    278
True     173
Name: count, dtype: int64

## Pivoting Past Results

In [350]:
# aggfunc is sum to account for races where two or more candidates from the same party are running in the general,
# we sum the votes of these candidates together as part of calculating two-party vote share
data = pd.pivot_table(data=res, values='candidatevotes', columns=['party_simplified'], index=['year', 'state', 'state_po', 'special'], aggfunc='sum').reset_index()
data.head()

party_simplified,year,state,state_po,special,DEMOCRAT,REPUBLICAN
0,2014,ALABAMA,AL,False,NaN,795606.0
1,2014,ALASKA,AK,False,129431.0,135445.0
2,2014,ARKANSAS,AR,False,334174.0,478819.0
3,2014,COLORADO,CO,False,944203.0,983891.0
4,2014,DELAWARE,DE,False,130655.0,98823.0


In [352]:
data.duplicated(subset=['year', 'state', 'special']).any() # np.False_ --> no duplicates

np.False_

In [360]:
# Get candidates

def get_cands(year, state, special, party):
    res_sorted = res.sort_values(by=['year', 'state', 'candidate'], ascending=True)
    
    df = res_sorted[
        (res_sorted['year'] == year) &
        (res_sorted['state'] == state) &
        (res_sorted['special'] == special) &
        (res_sorted['party_simplified'] == party)
    ]
    if df.shape[0] == 1:
        return df['candidate'].values[0]
    else:
        return repr(list(df['candidate'].values))

def get_incumbency_status(year, state, special, party):
    res_sorted = res.sort_values(by=['year', 'state', 'candidate'], ascending=True)
    
    df = res_sorted[
        (res_sorted['year'] == year) &
        (res_sorted['state'] == state) &
        (res_sorted['special'] == special) &
        (res_sorted['party_simplified'] == party)
    ]
    if df.shape[0] == 1:
        return df['inc'].values[0]
    else:
        return [bool(x) for x in df['inc'].values]

def get_ttl_receipts(year, state, special, party): # Misnomer: It's actually grabbing total individual contributions, not total receipts
    res_sorted = res.sort_values(by=['year', 'state', 'candidate'], ascending=True)
    
    df = res_sorted[
        (res_sorted['year'] == year) &
        (res_sorted['state'] == state) &
        (res_sorted['special'] == special) &
        (res_sorted['party_simplified'] == party)
    ]
    if df.shape[0] == 1:
        if np.isnan(df['TTL_INDIV_CONTRIB'].values[0]):
            return 0
        return df['TTL_INDIV_CONTRIB'].values[0]
    else:
        return [(0 if np.isnan(float(x)) else float(x)) for x in df['TTL_INDIV_CONTRIB'].values]

def get_totvotes(year, state, special):
    df = res[
        (res['year'] == year) &
        (res['state'] == state) &
        (res['special'] == special)
    ]
    return df['totalvotes'].values[0]

In [357]:
data['totalvotes'] = data[['year', 'state', 'special']].apply(lambda x: get_totvotes(x['year'], x['state'], x['special']), axis=1)
data.head()

party_simplified,year,state,state_po,special,DEMOCRAT,REPUBLICAN,totalvotes
0,2014,ALABAMA,AL,False,NaN,795606.0,818090.0
1,2014,ALASKA,AK,False,129431.0,135445.0,282400.0
2,2014,ARKANSAS,AR,False,334174.0,478819.0,847505.0
3,2014,COLORADO,CO,False,944203.0,983891.0,2041058.0
4,2014,DELAWARE,DE,False,130655.0,98823.0,234038.0


In [190]:
data.shape

(3047, 8)

In [361]:
def get_dem_cand(year, state, special):
    return get_cands(year, state, special, 'DEMOCRAT')

def get_rep_cand(year, state, special):
    return get_cands(year, state, special, 'REPUBLICAN')

def get_dem_ici(year, state, special):
    return get_incumbency_status(year, state, special, 'DEMOCRAT')

def get_rep_ici(year, state, special):
    return get_incumbency_status(year, state, special, 'REPUBLICAN')

def get_dem_receipts(year, state, special):
    return get_ttl_receipts(year, state, special, 'DEMOCRAT')

def get_rep_receipts(year, state, special):
    return get_ttl_receipts(year, state, special, 'REPUBLICAN')

In [362]:
data['dem_cand'] = data[['year', 'state', 'special']].apply(lambda x: get_dem_cand(x['year'], x['state'], x['special']), axis=1)
data['rep_cand'] = data[['year', 'state', 'special']].apply(lambda x: get_rep_cand(x['year'], x['state'], x['special']), axis=1)
data.head()

party_simplified,year,state,state_po,special,DEMOCRAT,REPUBLICAN,totalvotes,dem_cand,rep_cand
0,2014,ALABAMA,AL,False,NaN,795606.0,818090.0,[],JEFF SESSIONS
1,2014,ALASKA,AK,False,129431.0,135445.0,282400.0,MARK BEGICH,DAN SULLIVAN
2,2014,ARKANSAS,AR,False,334174.0,478819.0,847505.0,MARK L. PRYOR,TOM COTTON
3,2014,COLORADO,CO,False,944203.0,983891.0,2041058.0,MARK UDALL,CORY GARDNER
4,2014,DELAWARE,DE,False,130655.0,98823.0,234038.0,CHRISTOPHER A. COONS,KEVIN WADE


In [364]:
data['dem_inc'] = data[['year', 'state', 'special']].apply(lambda x: get_dem_ici(x['year'], x['state'], x['special']), axis=1)
data['rep_inc'] = data[['year', 'state', 'special']].apply(lambda x: get_rep_ici(x['year'], x['state'], x['special']), axis=1)
data.head()

party_simplified,year,state,state_po,special,DEMOCRAT,REPUBLICAN,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc
0,2014,ALABAMA,AL,False,NaN,795606.0,818090.0,[],JEFF SESSIONS,[],True
1,2014,ALASKA,AK,False,129431.0,135445.0,282400.0,MARK BEGICH,DAN SULLIVAN,True,False
2,2014,ARKANSAS,AR,False,334174.0,478819.0,847505.0,MARK L. PRYOR,TOM COTTON,True,False
3,2014,COLORADO,CO,False,944203.0,983891.0,2041058.0,MARK UDALL,CORY GARDNER,True,False
4,2014,DELAWARE,DE,False,130655.0,98823.0,234038.0,CHRISTOPHER A. COONS,KEVIN WADE,True,False


In [365]:
data['dem_funds'] = data[['year', 'state', 'special']].apply(lambda x: get_dem_receipts(x['year'], x['state'], x['special']), axis=1)
data['rep_funds'] = data[['year', 'state', 'special']].apply(lambda x: get_rep_receipts(x['year'], x['state'], x['special']), axis=1)
data.head()

party_simplified,year,state,state_po,special,DEMOCRAT,REPUBLICAN,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds
0,2014,ALABAMA,AL,False,NaN,795606.0,818090.0,[],JEFF SESSIONS,[],True,[],381899.0
1,2014,ALASKA,AK,False,129431.0,135445.0,282400.0,MARK BEGICH,DAN SULLIVAN,True,False,6020171.0,5966351.0
2,2014,ARKANSAS,AR,False,334174.0,478819.0,847505.0,MARK L. PRYOR,TOM COTTON,True,False,8000604.0,10892504.11
3,2014,COLORADO,CO,False,944203.0,983891.0,2041058.0,MARK UDALL,CORY GARDNER,True,False,13414189.0,8725449.0
4,2014,DELAWARE,DE,False,130655.0,98823.0,234038.0,CHRISTOPHER A. COONS,KEVIN WADE,True,False,2845059.0,92155.0


In [366]:
data = data.rename({'DEMOCRAT': 'dem', 'REPUBLICAN': 'rep'}, axis=1)
data.head()

party_simplified,year,state,state_po,special,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds
0,2014,ALABAMA,AL,False,NaN,795606.0,818090.0,[],JEFF SESSIONS,[],True,[],381899.0
1,2014,ALASKA,AK,False,129431.0,135445.0,282400.0,MARK BEGICH,DAN SULLIVAN,True,False,6020171.0,5966351.0
2,2014,ARKANSAS,AR,False,334174.0,478819.0,847505.0,MARK L. PRYOR,TOM COTTON,True,False,8000604.0,10892504.11
3,2014,COLORADO,CO,False,944203.0,983891.0,2041058.0,MARK UDALL,CORY GARDNER,True,False,13414189.0,8725449.0
4,2014,DELAWARE,DE,False,130655.0,98823.0,234038.0,CHRISTOPHER A. COONS,KEVIN WADE,True,False,2845059.0,92155.0


In [368]:
data['2party_votes'] = data['dem'] + data['rep']
data.head(2)

party_simplified,year,state,state_po,special,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes
0,2014,ALABAMA,AL,False,NaN,795606.0,818090.0,[],JEFF SESSIONS,[],True,[],381899.0,NaN
1,2014,ALASKA,AK,False,129431.0,135445.0,282400.0,MARK BEGICH,DAN SULLIVAN,True,False,6020171.0,5966351.0,264876.0


In [369]:
data['dem_cand'] = data['dem_cand'].str.title()
data['rep_cand'] = data['rep_cand'].str.title()
data.head(2)

party_simplified,year,state,state_po,special,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes
0,2014,ALABAMA,AL,False,NaN,795606.0,818090.0,[],Jeff Sessions,[],True,[],381899.0,NaN
1,2014,ALASKA,AK,False,129431.0,135445.0,282400.0,Mark Begich,Dan Sullivan,True,False,6020171.0,5966351.0,264876.0


In [370]:
data['state'] = data['state'].str.title()
data.head(2)

party_simplified,year,state,state_po,special,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes
0,2014,Alabama,AL,False,NaN,795606.0,818090.0,[],Jeff Sessions,[],True,[],381899.0,NaN
1,2014,Alaska,AK,False,129431.0,135445.0,282400.0,Mark Begich,Dan Sullivan,True,False,6020171.0,5966351.0,264876.0


In [371]:
# two-party vote share
data['dem_pct_2p'] = data['dem'] / data['2party_votes'] * 100
data['rep_pct_2p'] = data['rep'] / data['2party_votes'] * 100
data.head(2)

party_simplified,year,state,state_po,special,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p
0,2014,Alabama,AL,False,NaN,795606.0,818090.0,[],Jeff Sessions,[],True,[],381899.0,NaN,NaN,NaN
1,2014,Alaska,AK,False,129431.0,135445.0,282400.0,Mark Begich,Dan Sullivan,True,False,6020171.0,5966351.0,264876.0,48.864752,51.135248


In [372]:
data['dem_tot_funds'] = data['dem_funds'].map(
    lambda x: x if isinstance(x, float) else (np.sum(np.array(x)) if (isinstance(x, list) and len(x) > 0) else 0)
)
data['rep_tot_funds'] = data['rep_funds'].map(
    lambda x: x if isinstance(x, float) else (np.sum(np.array(x)) if (isinstance(x, list) and len(x) > 0) else 0)
)
data.head()

party_simplified,year,state,state_po,special,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p,dem_tot_funds,rep_tot_funds
0,2014,Alabama,AL,False,NaN,795606.0,818090.0,[],Jeff Sessions,[],True,[],381899.0,NaN,NaN,NaN,0.0,381899.00
1,2014,Alaska,AK,False,129431.0,135445.0,282400.0,Mark Begich,Dan Sullivan,True,False,6020171.0,5966351.0,264876.0,48.864752,51.135248,6020171.0,5966351.00
2,2014,Arkansas,AR,False,334174.0,478819.0,847505.0,Mark L. Pryor,Tom Cotton,True,False,8000604.0,10892504.11,812993.0,41.104167,58.895833,8000604.0,10892504.11
3,2014,Colorado,CO,False,944203.0,983891.0,2041058.0,Mark Udall,Cory Gardner,True,False,13414189.0,8725449.0,1928094.0,48.970797,51.029203,13414189.0,8725449.00
4,2014,Delaware,DE,False,130655.0,98823.0,234038.0,Christopher A. Coons,Kevin Wade,True,False,2845059.0,92155.0,229478.0,56.935741,43.064259,2845059.0,92155.00


In [373]:
data['dem_funds'] = data['dem_funds'].fillna(0)
data['rep_funds'] = data['rep_funds'].fillna(0)
data['tot_funds'] = data['dem_tot_funds'] + data['rep_tot_funds']
data['dem_funds_2p_pct'] = data['dem_tot_funds'] / data['tot_funds'] * 100
data['rep_funds_2p_pct'] = data['rep_tot_funds'] / data['tot_funds'] * 100
data.head(3)

party_simplified,year,state,state_po,special,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p,dem_tot_funds,rep_tot_funds,tot_funds,dem_funds_2p_pct,rep_funds_2p_pct
0,2014,Alabama,AL,False,NaN,795606.0,818090.0,[],Jeff Sessions,[],True,[],381899.0,NaN,NaN,NaN,0.0,381899.00,381899.00,0.000000,100.000000
1,2014,Alaska,AK,False,129431.0,135445.0,282400.0,Mark Begich,Dan Sullivan,True,False,6020171.0,5966351.0,264876.0,48.864752,51.135248,6020171.0,5966351.00,11986522.00,50.224502,49.775498
2,2014,Arkansas,AR,False,334174.0,478819.0,847505.0,Mark L. Pryor,Tom Cotton,True,False,8000604.0,10892504.11,812993.0,41.104167,58.895833,8000604.0,10892504.11,18893108.11,42.346680,57.653320


In [374]:
data[data['dem_funds_2p_pct'] == 50]

party_simplified,year,state,state_po,special,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p,dem_tot_funds,rep_tot_funds,tot_funds,dem_funds_2p_pct,rep_funds_2p_pct


In [375]:
data.shape

(207, 21)

In [376]:
data.to_csv('transformed/past_senate_results.csv', index_label=False)